**Linear Regression** is a supervised learning algorithm used to predict a continuous target variable ($y$) by modeling the relationship between one or more independent features ($X$).

**Simple Linear Regression (SLR)**

It is used when there is only one independent variable. The goal is to find the "Line of Best Fit."

The Equation:

$$y = wx + b$$

* $y$: Predicted output (Target)
* $w$: Slope (Weight/Coefficient)
* $x$: Input feature$b$: Intercept (Bias)

**Multiple Linear Regression (MLR)**

Multiple Linear Regression extends this concept to two or more independent variables. Instead of a 2D line, the algorithm fits a hyperplane in a multi-dimensional space.

The Equation:

$$y = w_1x_1 + w_2x_2 + ... + w_nx_n + b$$

In matrix form, this is represented concisely as:

$$\mathbf{y} = \mathbf{X}\mathbf{w}$$

* $\mathbf{X}$: Feature matrix (includes a column of $1$s for the bias)
* $\mathbf{w}$: Vector of weights (coefficients)

**Cost Function (Error Function)**

To determine the best weights, we must minimize the "distance" between the actual values and our predictions. We use the Mean Squared Error (MSE):

MSE=(1/n)​∑(yi​−yi​^​)2

where , 

* yi = actual value
* yi^ = predicted value
* n = number of samples

Goal ==> Minimise the error

**Ordinary Least Squares(OLS)**

For both SLR and MLR, the Ordinary Least Squares (OLS) method provides an analytical solution to find the optimal weights ($\hat{w}$) directly using matrix algebra:

$$\hat{\beta} = (X^T X)^{-1} X^T y$$

Note: This method is highly efficient for smaller datasets but becomes computationally expensive ($O(n^3)$) as the number of features ($n$) grows very large.

**Data Loading**

We'll load the Diabetes dataset, which contains 442 patients with 10 baseline variables (age, sex, bmi, etc.).

In [4]:
import numpy as np
import pandas as pd
from sklearn import datasets
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

diabetes = datasets.load_diabetes()
X = diabetes.data
y = diabetes.target.reshape(-1, 1) 

print(f"Dataset loaded with {X.shape[0]} samples and {X.shape[1]} features.")

Dataset loaded with 442 samples and 10 features.


**Custom OLS Implementation**

In [5]:

# 1. Add a column of ones to X for the intercept term (bias)
# This transforms X from [x1, x2...] to [[1, x1], [1, x2]...]
X_b = np.c_[np.ones((X.shape[0], 1)), X]
    
# 2. Calculate the Normal Equation: (X^T * X)^-1 * X^T * y
# np.linalg.inv computes the inverse
# .T is the transpose
# .dot() or @ is matrix multiplication
weights = np.linalg.inv(X_b.T.dot(X_b)).dot(X_b.T).dot(y)
    
intercept = weights[0][0]
coefficients = weights[1:].flatten()


print("Manual OLS Fit Complete.")

Manual OLS Fit Complete.


**Sklearn Implementation**

In [6]:
model = LinearRegression()
model.fit(X, y)

sklearn_intercept = model.intercept_[0]
sklearn_coeffs = model.coef_.flatten()

print("Sklearn Fit Complete.")

Sklearn Fit Complete.


In [7]:

y_pred_manual = X_b @ weights

y_pred_sklearn = model.predict(X)

metrics_data = {
    "Metric": ["MAE", "MSE", "RMSE", "R2 Score"],
    "Manual OLS": [
        mean_absolute_error(y, y_pred_manual),
        mean_squared_error(y, y_pred_manual),
        np.sqrt(mean_squared_error(y, y_pred_manual)),
        r2_score(y, y_pred_manual)
    ],
    "Sklearn": [
        mean_absolute_error(y, y_pred_sklearn),
        mean_squared_error(y, y_pred_sklearn),
        np.sqrt(mean_squared_error(y, y_pred_sklearn)),
        r2_score(y, y_pred_sklearn)
    ]
}

metrics_df = pd.DataFrame(metrics_data)
metrics_df["Difference"] = metrics_df["Manual OLS"] - metrics_df["Sklearn"]

print("--- Model Performance Comparison ---")
print(metrics_df.to_string(index=False))

--- Model Performance Comparison ---
  Metric  Manual OLS     Sklearn    Difference
     MAE   43.277452   43.277452  0.000000e+00
     MSE 2859.696348 2859.696348  4.547474e-13
    RMSE   53.476129   53.476129  7.105427e-15
R2 Score    0.517748    0.517748 -1.110223e-16


Why it worked: Diabetes dataset had only 10 features and 442 rows. The matrix $(X^T X)$ was only $11 \times 11$, which is incredibly easy for our computer to invert instantly.

The Limitation: If we had 20,000 features in a complex dataset, that np.linalg.inv step would become very slow and might crash the memory.

Why Sklearn? While your manual code works, sklearn.LinearRegression is more robust. It handles edge cases, like when your features are perfectly correlated, by using **SVD (Singular Value Decomposition)** instead of basic matrix inversion.